# Current SOXL Model vs Buy-and-Hold YTD / June 2026

This executable diagnostic composes the current production runtime model (`v19reg_v22_budget_usage_bull_guard_v1`) from the refreshed slow/fast daily paths and compares it with SOXL buy-and-hold from 2026-01-01 and during June 2026.

The production data panel is fresh through 2026-06-10, while dashboard model replays are safely backtestable through 2026-06-09. The comparison therefore uses 2026-06-09 as the common end date and records the 2026-06-10 market-only buy-and-hold state separately.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

OUTPUT_DIR = ROOT / "runtime/production/card/current_model_ytd_june_comparison_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROFILE = "v19reg_v22_budget_usage_bull_guard_v1"
SLOW_PROFILE = "balanced_rule_residual_v19_reg"
FAST_PROFILE = "balanced_rule_residual_v22"
SLOW_DAILY = ROOT / f"runtime/model/card/{SLOW_PROFILE}/{SLOW_PROFILE}_soxl_daily.csv"
FAST_DAILY = ROOT / f"runtime/model/card/{FAST_PROFILE}/{FAST_PROFILE}_soxl_daily.csv"
PANEL_PATH = ROOT / "data/processed/panels/soxl_soxx_vix_full.csv"
FRESHNESS_PATH = ROOT / "dashboard_data/model_replay_freshness.json"
LATEST_SIGNAL_SUMMARY_PATH = ROOT / "runtime/soxl/paper_trading/2026-06-10/summary.json"

for required in [SLOW_DAILY, FAST_DAILY, PANEL_PATH, FRESHNESS_PATH]:
    if not required.exists():
        raise FileNotFoundError(required)

print(f"repo={ROOT}")
print(f"output_dir={OUTPUT_DIR.relative_to(ROOT)}")

repo=/Users/rziiiii/dev/workspace
output_dir=runtime/production/card/current_model_ytd_june_comparison_v1


In [2]:
from src.soxl_quant.app.budget_counterfactual import (
    BullParticipationGuardConfig,
    build_budget_usage_counterfactual,
    dashboard_daily_frame,
)
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

panel = pd.read_csv(PANEL_PATH, parse_dates=["date"]).sort_values("date")
freshness = json.loads(FRESHNESS_PATH.read_text(encoding="utf-8"))
latest_signal_summary = (
    json.loads(LATEST_SIGNAL_SUMMARY_PATH.read_text(encoding="utf-8"))
    if LATEST_SIGNAL_SUMMARY_PATH.exists()
    else {}
)

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset()
price = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
feature_frame = dataset.handler.feature_frame.droplevel("instrument").sort_index()

slow = pd.read_csv(SLOW_DAILY)
fast = pd.read_csv(FAST_DAILY)

result = build_budget_usage_counterfactual(
    slow,
    fast,
    price,
    slow_label=SLOW_PROFILE,
    fast_label=FAST_PROFILE,
    composed_label=PROFILE,
    profile=PROFILE,
    description=(
        "Current production runtime composer rebuilt from refreshed dashboard model replays. "
        "The slow v19_reg actor sets the risk mode and the v22 actor provides usage inside the mode band."
    ),
    bull_guard=BullParticipationGuardConfig(enabled=True),
    feature_frame=feature_frame,
    windows={
        "ytd_2026_common": ("2026-01-01", str(freshness.get("safe_backtest_end", "2026-06-09"))),
        "june_2026_common": ("2026-06-01", str(freshness.get("safe_backtest_end", "2026-06-09"))),
    },
)

model_daily = dashboard_daily_frame(result.daily, price)
model_daily["datetime"] = pd.to_datetime(model_daily["datetime"])

model_daily_path = OUTPUT_DIR / "current_model_daily.csv"
model_daily.to_csv(model_daily_path, index=False)
composition_payload_path = OUTPUT_DIR / "current_model_composition_payload.json"
composition_payload_path.write_text(json.dumps(result.payload, indent=2), encoding="utf-8")

print(f"canonical_panel_last_date={panel['date'].max().date()}")
print(f"dashboard_safe_backtest_end={freshness.get('safe_backtest_end')}")
print(f"model_daily_start={model_daily['datetime'].min().date()} model_daily_end={model_daily['datetime'].max().date()} rows={len(model_daily)}")
print(f"latest_signal_date={latest_signal_summary.get('date')} target={latest_signal_summary.get('signal', {}).get('target_position')}")

[37550:MainThread](2026-06-11 15:37:40,880) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[37550:MainThread](2026-06-11 15:37:41,301) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[37550:MainThread](2026-06-11 15:37:41,301) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[37550:MainThread](2026-06-11 15:37:42,600) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[37550:MainThread](2026-06-11 15:37:42,601) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[37550:MainThread](2026-06-11 15:37:42,601) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


canonical_panel_last_date=2026-06-10
dashboard_safe_backtest_end=2026-06-09
model_daily_start=2020-01-02 model_daily_end=2026-06-09 rows=1617
latest_signal_date=2026-06-10 target=0.205


In [3]:
def _max_drawdown_pct(nav: pd.Series) -> float:
    rebased = nav.astype(float) / float(nav.iloc[0])
    drawdown = rebased / rebased.cummax() - 1.0
    return abs(float(drawdown.min()) * 100.0)


def _window_metrics(frame: pd.DataFrame, label: str, start: str, end: str) -> dict:
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    sub = frame[(frame["datetime"] >= start_ts) & (frame["datetime"] <= end_ts)].copy()
    if len(sub) < 2:
        raise ValueError(f"{label} has fewer than 2 rows between {start} and {end}")
    sub = sub.sort_values("datetime").reset_index(drop=True)
    strategy_nav = sub["strategy_nav"].astype(float)
    buyhold_nav = sub["buyhold_nav"].astype(float)
    positions = sub["position"].astype(float)
    position_changes = positions.diff().abs().fillna(0.0)
    return {
        "window": label,
        "start": sub["datetime"].iloc[0].date().isoformat(),
        "end": sub["datetime"].iloc[-1].date().isoformat(),
        "rows": int(len(sub)),
        "soxl_buyhold_return_pct": float((buyhold_nav.iloc[-1] / buyhold_nav.iloc[0] - 1.0) * 100.0),
        "current_model_return_pct": float((strategy_nav.iloc[-1] / strategy_nav.iloc[0] - 1.0) * 100.0),
        "excess_return_pct": float(
            (strategy_nav.iloc[-1] / strategy_nav.iloc[0] - 1.0) * 100.0
            - (buyhold_nav.iloc[-1] / buyhold_nav.iloc[0] - 1.0) * 100.0
        ),
        "soxl_buyhold_max_drawdown_pct": _max_drawdown_pct(buyhold_nav),
        "current_model_max_drawdown_pct": _max_drawdown_pct(strategy_nav),
        "average_position_pct": float(positions.mean() * 100.0),
        "end_position_pct": float(positions.iloc[-1] * 100.0),
        "position_change_count": int((position_changes > 0.001).sum()),
    }


def _market_only_metrics(panel_frame: pd.DataFrame, label: str, start: str, end: str) -> dict:
    sub = panel_frame[
        (panel_frame["date"] >= pd.Timestamp(start)) & (panel_frame["date"] <= pd.Timestamp(end))
    ].copy()
    if len(sub) < 2:
        raise ValueError(f"{label} has fewer than 2 market rows between {start} and {end}")
    nav = sub["soxl_close"].astype(float) / float(sub["soxl_close"].iloc[0])
    return {
        "window": label,
        "start": sub["date"].iloc[0].date().isoformat(),
        "end": sub["date"].iloc[-1].date().isoformat(),
        "rows": int(len(sub)),
        "soxl_buyhold_return_pct": float((nav.iloc[-1] - 1.0) * 100.0),
        "soxl_buyhold_max_drawdown_pct": _max_drawdown_pct(nav),
    }


def _normalise_daily_frame(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    if "datetime" not in frame.columns:
        raise ValueError(f"daily CSV missing datetime: {path}")
    required = {"strategy_nav", "buyhold_nav", "price", "position"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"daily CSV missing {sorted(missing)}: {path}")
    frame = frame.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    return frame.sort_values("datetime").reset_index(drop=True)


def _model_window_metrics(
    frame: pd.DataFrame,
    *,
    profile: str,
    status: str,
    model_kind: str,
    source_path: Path,
    window: str,
    start: str,
    end: str,
) -> dict:
    metrics = _window_metrics(frame, window, start, end)
    return {
        "profile": profile,
        "status": status,
        "model_kind": model_kind,
        "source_path": source_path.as_posix(),
        **metrics,
    }


common_end = min(
    pd.Timestamp(str(freshness.get("safe_backtest_end"))).date(),
    model_daily["datetime"].max().date(),
)
latest_data_end = panel["date"].max().date()

comparison_rows = [
    _window_metrics(model_daily, "ytd_2026_common", "2026-01-01", common_end.isoformat()),
    _window_metrics(model_daily, "june_2026_common", "2026-06-01", common_end.isoformat()),
]
metrics_df = pd.DataFrame(comparison_rows)
metrics_path = OUTPUT_DIR / "comparison_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

market_only_rows = [
    _market_only_metrics(panel, "ytd_2026_market_latest", "2026-01-01", latest_data_end.isoformat()),
    _market_only_metrics(panel, "june_2026_market_latest", "2026-06-01", latest_data_end.isoformat()),
]
market_only_df = pd.DataFrame(market_only_rows)
market_only_path = OUTPUT_DIR / "market_only_latest_metrics.csv"
market_only_df.to_csv(market_only_path, index=False)

model_specs = []
for item in freshness.get("profiles", []):
    output = ROOT / item["output"]
    if not output.exists():
        raise FileNotFoundError(output)
    model_specs.append(
        {
            "profile": item["profile"],
            "status": item.get("status", ""),
            "model_kind": "dashboard_replay",
            "source_path": output.relative_to(ROOT),
            "daily": _normalise_daily_frame(output),
        }
    )
model_specs.append(
    {
        "profile": PROFILE,
        "status": "production_composer",
        "model_kind": "runtime_composer",
        "source_path": model_daily_path.relative_to(ROOT),
        "daily": model_daily.copy(),
    }
)

all_model_daily_frames = []
all_model_source_rows = []
for spec in model_specs:
    daily = spec["daily"].copy()
    daily["profile"] = spec["profile"]
    daily["status"] = spec["status"]
    daily["model_kind"] = spec["model_kind"]
    daily["source_path"] = spec["source_path"].as_posix()
    all_model_daily_frames.append(
        daily[
            [
                "profile",
                "status",
                "model_kind",
                "source_path",
                "datetime",
                "strategy_nav",
                "price",
                "buyhold_nav",
                "position",
                "excess",
                "excess_pct",
                "action",
            ]
            + [col for col in daily.columns if col not in {
                "profile",
                "status",
                "model_kind",
                "source_path",
                "datetime",
                "strategy_nav",
                "price",
                "buyhold_nav",
                "position",
                "excess",
                "excess_pct",
                "action",
            }]
        ]
    )
    all_model_source_rows.append(
        {
            "profile": spec["profile"],
            "status": spec["status"],
            "model_kind": spec["model_kind"],
            "source_path": spec["source_path"].as_posix(),
            "start": daily["datetime"].min().date().isoformat(),
            "end": daily["datetime"].max().date().isoformat(),
            "rows": int(len(daily)),
        }
    )

all_model_daily_df = pd.concat(all_model_daily_frames, ignore_index=True)
all_model_daily_path = OUTPUT_DIR / "all_model_daily_nav.csv"
all_model_daily_df.to_csv(all_model_daily_path, index=False)

all_model_source_df = pd.DataFrame(all_model_source_rows)
all_model_source_path = OUTPUT_DIR / "all_model_source_paths.csv"
all_model_source_df.to_csv(all_model_source_path, index=False)

all_model_metric_rows = []
for spec in model_specs:
    source_path = spec["source_path"]
    all_model_metric_rows.append(
        _model_window_metrics(
            spec["daily"],
            profile=spec["profile"],
            status=spec["status"],
            model_kind=spec["model_kind"],
            source_path=source_path,
            window="ytd_2026_common",
            start="2026-01-01",
            end=common_end.isoformat(),
        )
    )
    all_model_metric_rows.append(
        _model_window_metrics(
            spec["daily"],
            profile=spec["profile"],
            status=spec["status"],
            model_kind=spec["model_kind"],
            source_path=source_path,
            window="june_2026_common",
            start="2026-06-01",
            end=common_end.isoformat(),
        )
    )

all_model_metrics_df = pd.DataFrame(all_model_metric_rows)
all_model_metrics_df = all_model_metrics_df.sort_values(
    ["window", "current_model_return_pct", "current_model_max_drawdown_pct"],
    ascending=[True, False, True],
).reset_index(drop=True)
all_model_metrics_path = OUTPUT_DIR / "all_model_comparison_metrics.csv"
all_model_metrics_df.to_csv(all_model_metrics_path, index=False)

rank_cols = [
    "window",
    "profile",
    "status",
    "model_kind",
    "start",
    "end",
    "current_model_return_pct",
    "excess_return_pct",
    "current_model_max_drawdown_pct",
    "average_position_pct",
    "end_position_pct",
]

print("Common-window comparison through", common_end.isoformat())
display_cols = [
    "window",
    "start",
    "end",
    "soxl_buyhold_return_pct",
    "current_model_return_pct",
    "excess_return_pct",
    "soxl_buyhold_max_drawdown_pct",
    "current_model_max_drawdown_pct",
    "average_position_pct",
    "end_position_pct",
]
display(metrics_df[display_cols].style.format({
    "soxl_buyhold_return_pct": "{:.2f}",
    "current_model_return_pct": "{:.2f}",
    "excess_return_pct": "{:.2f}",
    "soxl_buyhold_max_drawdown_pct": "{:.2f}",
    "current_model_max_drawdown_pct": "{:.2f}",
    "average_position_pct": "{:.2f}",
    "end_position_pct": "{:.2f}",
}))

print("All model rows:", len(model_specs), "models x 2 windows =", len(all_model_metrics_df), "metric rows")
print("YTD 2026 ranking")
display(
    all_model_metrics_df[all_model_metrics_df["window"] == "ytd_2026_common"][rank_cols]
    .sort_values("current_model_return_pct", ascending=False)
    .style.format({
        "current_model_return_pct": "{:.2f}",
        "excess_return_pct": "{:.2f}",
        "current_model_max_drawdown_pct": "{:.2f}",
        "average_position_pct": "{:.2f}",
        "end_position_pct": "{:.2f}",
    })
)
print("June 2026 ranking")
display(
    all_model_metrics_df[all_model_metrics_df["window"] == "june_2026_common"][rank_cols]
    .sort_values("current_model_return_pct", ascending=False)
    .style.format({
        "current_model_return_pct": "{:.2f}",
        "excess_return_pct": "{:.2f}",
        "current_model_max_drawdown_pct": "{:.2f}",
        "average_position_pct": "{:.2f}",
        "end_position_pct": "{:.2f}",
    })
)

print("Market-only SOXL latest-data readout through", latest_data_end.isoformat())
display(market_only_df.style.format({
    "soxl_buyhold_return_pct": "{:.2f}",
    "soxl_buyhold_max_drawdown_pct": "{:.2f}",
}))

Common-window comparison through 2026-06-09


,window,start,end,soxl_buyhold_return_pct,current_model_return_pct,excess_return_pct,soxl_buyhold_max_drawdown_pct,current_model_max_drawdown_pct,average_position_pct,end_position_pct
0,ytd_2026_common,2026-01-02,2026-06-09,326.93,3.25,-323.68,43.47,21.09,28.53,0.00
1,june_2026_common,2026-06-01,2026-06-09,-11.17,0.00,11.17,34.93,0.00,0.00,0.00


All model rows: 14 models x 2 windows = 28 metric rows
YTD 2026 ranking


,window,profile,status,model_kind,start,end,current_model_return_pct,excess_return_pct,current_model_max_drawdown_pct,average_position_pct,end_position_pct
14,ytd_2026_common,smoke,canonical,dashboard_replay,2026-01-02,2026-06-09,224.36,-102.56,30.35,34.24,0.00
15,ytd_2026_common,balanced_macro_v1,candidate,dashboard_replay,2026-01-02,2026-06-09,196.75,-130.18,29.90,33.32,0.00
16,ytd_2026_common,state_conditioned_drawdown_rebound_reward_train_v1,ablation,dashboard_replay,2026-01-02,2026-06-09,196.51,-130.42,31.39,31.80,0.00
17,ytd_2026_common,mode_conditioned_rebound_reward_train_v2,ablation,dashboard_replay,2026-01-02,2026-06-09,177.57,-149.35,27.28,29.40,0.00
18,ytd_2026_common,balanced_rule_residual_v19_reg_v22_retest_v1,ablation,dashboard_replay,2026-01-02,2026-06-09,175.01,-151.91,27.12,29.71,0.00
19,ytd_2026_common,balanced_rule_residual_v19,archived,dashboard_replay,2026-01-02,2026-06-09,173.30,-153.63,24.40,28.52,0.00
20,ytd_2026_common,drawdown_rebound_reward_train_v1,ablation,dashboard_replay,2026-01-02,2026-06-09,172.83,-154.09,27.31,29.41,0.00
21,ytd_2026_common,balanced_rule_residual_v19_reg_light,candidate,dashboard_replay,2026-01-02,2026-06-09,164.25,-162.67,25.99,28.12,0.00
22,ytd_2026_common,balanced,archived,dashboard_replay,2026-01-02,2026-06-09,159.21,-167.72,29.70,30.42,0.00
23,ytd_2026_common,balanced_rule_residual_imitation_v4,candidate,dashboard_replay,2026-01-02,2026-06-09,145.03,-181.90,26.00,26.60,0.00


June 2026 ranking


,window,profile,status,model_kind,start,end,current_model_return_pct,excess_return_pct,current_model_max_drawdown_pct,average_position_pct,end_position_pct
0,june_2026_common,v19reg_v22_budget_usage_bull_guard_v1,production_composer,runtime_composer,2026-06-01,2026-06-09,0.00,11.17,0.00,0.00,0.00
1,june_2026_common,balanced_rule_residual_v19,archived,dashboard_replay,2026-06-01,2026-06-09,-7.28,3.88,24.40,0.00,0.00
2,june_2026_common,balanced_rule_residual_v19_reg,candidate,dashboard_replay,2026-06-01,2026-06-09,-7.31,3.85,24.49,0.00,0.00
3,june_2026_common,balanced_rule_residual_v19_reg_light,candidate,dashboard_replay,2026-06-01,2026-06-09,-7.54,3.63,25.14,0.00,0.00
4,june_2026_common,balanced_macro_v1,candidate,dashboard_replay,2026-06-01,2026-06-09,-7.72,3.45,29.90,0.00,0.00
5,june_2026_common,balanced_rule_residual_imitation_v4,candidate,dashboard_replay,2026-06-01,2026-06-09,-7.84,3.33,26.00,0.00,0.00
6,june_2026_common,balanced_rule_residual_v22,canonical,dashboard_replay,2026-06-01,2026-06-09,-7.94,3.23,26.28,0.00,0.00
7,june_2026_common,balanced_rule_residual_v22_continued_retention_v1,ablation,dashboard_replay,2026-06-01,2026-06-09,-7.94,3.23,26.28,0.00,0.00
8,june_2026_common,balanced_rule_residual_v19_reg_v22_retest_v1,ablation,dashboard_replay,2026-06-01,2026-06-09,-8.10,3.07,26.73,0.00,0.00
9,june_2026_common,smoke,canonical,dashboard_replay,2026-06-01,2026-06-09,-8.22,2.95,27.77,0.00,0.00


Market-only SOXL latest-data readout through 2026-06-10


,window,start,end,rows,soxl_buyhold_return_pct,soxl_buyhold_max_drawdown_pct
0,ytd_2026_market_latest,2026-01-02,2026-06-10,110,282.41,43.47
1,june_2026_market_latest,2026-06-01,2026-06-10,8,-20.43,35.61


In [4]:
def _json_safe(value):
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [_json_safe(v) for v in value]
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value

summary = {
    "created_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(),
    "experiment": "current_model_ytd_june_comparison_v1",
    "direction": "production",
    "profile": PROFILE,
    "slow_profile": SLOW_PROFILE,
    "fast_profile": FAST_PROFILE,
    "comparison_common_end": common_end.isoformat(),
    "canonical_panel_last_date": latest_data_end.isoformat(),
    "dashboard_model_replay_freshness": freshness,
    "latest_signal_summary": {
        "path": str(LATEST_SIGNAL_SUMMARY_PATH.relative_to(ROOT)) if LATEST_SIGNAL_SUMMARY_PATH.exists() else None,
        "date": latest_signal_summary.get("date"),
        "profile": latest_signal_summary.get("profile"),
        "broker": latest_signal_summary.get("broker"),
        "dry_run": latest_signal_summary.get("dry_run"),
        "executed": latest_signal_summary.get("executed"),
        "target_position": latest_signal_summary.get("signal", {}).get("target_position"),
        "risk_gate_approved": latest_signal_summary.get("risk_gate", {}).get("approved"),
    },
    "scheduler_fix_readout": {
        "launchagent_plists_synced": True,
        "scheduled_runtime_composer": "budget_usage",
        "scheduled_profile": PROFILE,
        "postclose_manual_run_result": "completed_with_local_paper_dry_run_after_futu_opend_unreachable",
        "futu_opend_status": "unreachable_127.0.0.1_11111_at_manual_verification",
        "data_update_status": "fresh_no_new_rows_expected_last_trading_day_2026-06-10",
    },
    "metrics_common_window": metrics_df.to_dict(orient="records"),
    "all_model_source_paths": all_model_source_df.to_dict(orient="records"),
    "all_model_metrics_common_window": all_model_metrics_df.to_dict(orient="records"),
    "market_only_latest_metrics": market_only_df.to_dict(orient="records"),
    "artifacts": {
        "notebook": "runtime/production/card/current_model_ytd_june_comparison_v1/current_model_ytd_june_comparison_v1.ipynb",
        "current_model_daily": str(model_daily_path.relative_to(ROOT)),
        "composition_payload": str(composition_payload_path.relative_to(ROOT)),
        "comparison_metrics": str(metrics_path.relative_to(ROOT)),
        "all_model_daily_nav": str(all_model_daily_path.relative_to(ROOT)),
        "all_model_source_paths": str(all_model_source_path.relative_to(ROOT)),
        "all_model_comparison_metrics": str(all_model_metrics_path.relative_to(ROOT)),
        "market_only_latest_metrics": str(market_only_path.relative_to(ROOT)),
        "summary": "runtime/production/card/current_model_ytd_june_comparison_v1/summary.json",
    },
}
summary = _json_safe(summary)
summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps({
    "summary": str(summary_path.relative_to(ROOT)),
    "common_end": summary["comparison_common_end"],
    "canonical_panel_last_date": summary["canonical_panel_last_date"],
    "model_count": len(all_model_source_df),
    "all_model_metric_rows": len(all_model_metrics_df),
    "all_model_comparison_metrics": str(all_model_metrics_path.relative_to(ROOT)),
    "all_model_daily_nav": str(all_model_daily_path.relative_to(ROOT)),
}, indent=2))

{
  "summary": "runtime/production/card/current_model_ytd_june_comparison_v1/summary.json",
  "common_end": "2026-06-09",
  "canonical_panel_last_date": "2026-06-10",
  "model_count": 14,
  "all_model_metric_rows": 28,
  "all_model_comparison_metrics": "runtime/production/card/current_model_ytd_june_comparison_v1/all_model_comparison_metrics.csv",
  "all_model_daily_nav": "runtime/production/card/current_model_ytd_june_comparison_v1/all_model_daily_nav.csv"
}


In [5]:
from IPython.display import Markdown, display

ytd = metrics_df.loc[metrics_df["window"] == "ytd_2026_common"].iloc[0]
june = metrics_df.loc[metrics_df["window"] == "june_2026_common"].iloc[0]
ytd_market_latest = market_only_df.loc[market_only_df["window"] == "ytd_2026_market_latest"].iloc[0]
june_market_latest = market_only_df.loc[market_only_df["window"] == "june_2026_market_latest"].iloc[0]
latest_signal = summary["latest_signal_summary"]

ytd_ranked = all_model_metrics_df[all_model_metrics_df["window"] == "ytd_2026_common"].sort_values(
    "current_model_return_pct", ascending=False
)
june_ranked = all_model_metrics_df[all_model_metrics_df["window"] == "june_2026_common"].sort_values(
    "current_model_return_pct", ascending=False
)
ytd_best = ytd_ranked.iloc[0]
ytd_worst = ytd_ranked.iloc[-1]
june_best = june_ranked.iloc[0]
june_worst = june_ranked.iloc[-1]

conclusion = f"""
## Conclusion

- Scheduler/data fix status: LaunchAgent paths now point at this repo and scheduled preopen/postclose runs use the current `{PROFILE}` runtime composer. The manual postclose run refreshed data/dashboard successfully; Futu OpenD was unreachable on `127.0.0.1:11111`, so the scheduler retried as a local paper dry-run and still produced the current signal artifact.
- Data status: canonical SOXL/SOXX/VIX panel is fresh through `{latest_data_end.isoformat()}`; dashboard model replays are safe through `{common_end.isoformat()}`. The model-vs-SOXL comparison below is therefore apples-to-apples through `{common_end.isoformat()}`.
- All-model data status: `{len(all_model_source_df)}` model paths are included: `{len(freshness.get('profiles', []))}` dashboard replay models plus the current production composer. The full daily data is in `all_model_daily_nav.csv`; the YTD/June ranking is in `all_model_comparison_metrics.csv`.
- Current production composer, YTD 2026 common window (`{ytd['start']}` to `{ytd['end']}`): SOXL buy-and-hold `{ytd['soxl_buyhold_return_pct']:.2f}%`; current model `{ytd['current_model_return_pct']:.2f}%`; excess `{ytd['excess_return_pct']:.2f}pp`; model max drawdown `{ytd['current_model_max_drawdown_pct']:.2f}%` vs SOXL `{ytd['soxl_buyhold_max_drawdown_pct']:.2f}%`.
- Current production composer, June 2026 common window (`{june['start']}` to `{june['end']}`): SOXL buy-and-hold `{june['soxl_buyhold_return_pct']:.2f}%`; current model `{june['current_model_return_pct']:.2f}%`; excess `{june['excess_return_pct']:.2f}pp`; model max drawdown `{june['current_model_max_drawdown_pct']:.2f}%` vs SOXL `{june['soxl_buyhold_max_drawdown_pct']:.2f}%`.
- All-model YTD leader: `{ytd_best['profile']}` `{ytd_best['current_model_return_pct']:.2f}%` return, `{ytd_best['current_model_max_drawdown_pct']:.2f}%` max drawdown. YTD laggard: `{ytd_worst['profile']}` `{ytd_worst['current_model_return_pct']:.2f}%` return.
- All-model June leader: `{june_best['profile']}` `{june_best['current_model_return_pct']:.2f}%` return, `{june_best['current_model_max_drawdown_pct']:.2f}%` max drawdown. June laggard: `{june_worst['profile']}` `{june_worst['current_model_return_pct']:.2f}%` return.
- Market-only latest readout through `{latest_data_end.isoformat()}`: SOXL buy-and-hold is `{ytd_market_latest['soxl_buyhold_return_pct']:.2f}%` YTD and `{june_market_latest['soxl_buyhold_return_pct']:.2f}%` for June.
- Latest current signal artifact: `{latest_signal.get('date')}` profile `{latest_signal.get('profile')}`, dry_run=`{latest_signal.get('dry_run')}`, executed=`{latest_signal.get('executed')}`, target_position=`{latest_signal.get('target_position')}`, risk_gate_approved=`{latest_signal.get('risk_gate_approved')}`.
- Decision: scheduler/data/dashboard path is unblocked for local evidence generation, but real Futu execution proof still needs OpenD back online. In the common windows, the current production composer severely lagged SOXL YTD because SOXL rallied while the model stayed defensive, but it protected the June selloff by staying flat.
""".strip()

conclusion_path = OUTPUT_DIR / "conclusion.md"
conclusion_path.write_text(conclusion + "\n", encoding="utf-8")
display(Markdown(conclusion))

## Conclusion

- Scheduler/data fix status: LaunchAgent paths now point at this repo and scheduled preopen/postclose runs use the current `v19reg_v22_budget_usage_bull_guard_v1` runtime composer. The manual postclose run refreshed data/dashboard successfully; Futu OpenD was unreachable on `127.0.0.1:11111`, so the scheduler retried as a local paper dry-run and still produced the current signal artifact.
- Data status: canonical SOXL/SOXX/VIX panel is fresh through `2026-06-10`; dashboard model replays are safe through `2026-06-09`. The model-vs-SOXL comparison below is therefore apples-to-apples through `2026-06-09`.
- All-model data status: `14` model paths are included: `13` dashboard replay models plus the current production composer. The full daily data is in `all_model_daily_nav.csv`; the YTD/June ranking is in `all_model_comparison_metrics.csv`.
- Current production composer, YTD 2026 common window (`2026-01-02` to `2026-06-09`): SOXL buy-and-hold `326.93%`; current model `3.25%`; excess `-323.68pp`; model max drawdown `21.09%` vs SOXL `43.47%`.
- Current production composer, June 2026 common window (`2026-06-01` to `2026-06-09`): SOXL buy-and-hold `-11.17%`; current model `0.00%`; excess `11.17pp`; model max drawdown `0.00%` vs SOXL `34.93%`.
- All-model YTD leader: `smoke` `224.36%` return, `30.35%` max drawdown. YTD laggard: `v19reg_v22_budget_usage_bull_guard_v1` `3.25%` return.
- All-model June leader: `v19reg_v22_budget_usage_bull_guard_v1` `0.00%` return, `0.00%` max drawdown. June laggard: `state_conditioned_drawdown_rebound_reward_train_v1` `-9.80%` return.
- Market-only latest readout through `2026-06-10`: SOXL buy-and-hold is `282.41%` YTD and `-20.43%` for June.
- Latest current signal artifact: `2026-06-10` profile `v19reg_v22_budget_usage_bull_guard_v1`, dry_run=`True`, executed=`False`, target_position=`0.205`, risk_gate_approved=`True`.
- Decision: scheduler/data/dashboard path is unblocked for local evidence generation, but real Futu execution proof still needs OpenD back online. In the common windows, the current production composer severely lagged SOXL YTD because SOXL rallied while the model stayed defensive, but it protected the June selloff by staying flat.